In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np

path = '/content/drive/MyDrive/Data Analyst/EduSmart/data/cleaned'

Mounted at /content/drive


In [ ]:
dim_courses = pd.read_csv(f'{path}/dim_courses_cleaned.csv')
fact_enrollments = pd.read_csv(f'{path}/fact_enrollments_cleaned.csv')
fact_progress = pd.read_csv(f'{path}/fact_progress_cleaned.csv')

# Konversi tanggal
fact_enrollments['enrollment_date'] = pd.to_datetime(fact_enrollments['enrollment_date'])
fact_progress['completion_date'] = pd.to_datetime(fact_progress['completion_date'])

print(f"fact_enrollments: {len(fact_enrollments)} baris")
print(f"fact_progress: {len(fact_progress)} baris")

fact_enrollments: 7947 baris
fact_progress: 35526 baris


In [ ]:
# === Feature: progress_speed_week1 ===
# Jumlah modul yang berhasil diselesaikan dalam 7 hari pertama sejak enrollment_date

# Gabungkan completion_date modul dengan enrollment_date
progress_with_enroll = fact_progress.merge(
    fact_enrollments[['enrollment_id', 'enrollment_date']],
    on='enrollment_id'
)

# Hitung selisih hari dari enrollment ke completion tiap modul
progress_with_enroll['hari_ke_completion'] = (
    progress_with_enroll['completion_date'] - progress_with_enroll['enrollment_date']
).dt.days

# Filter: modul yang completed DALAM 7 hari pertama (hari 0-7)
modul_week1 = progress_with_enroll[
    (progress_with_enroll['status'] == 'completed') &
    (progress_with_enroll['hari_ke_completion'] >= 0) &
    (progress_with_enroll['hari_ke_completion'] <= 7)
]

# Agregasi per enrollment
progress_speed_week1 = modul_week1.groupby('enrollment_id').size().rename('progress_speed_week1').reset_index()

print(f"Enrollment dengan aktivitas di minggu pertama: {len(progress_speed_week1)}")
print(progress_speed_week1['progress_speed_week1'].describe())

Enrollment dengan aktivitas di minggu pertama: 4133
count    4133.000000
mean        1.443020
std         0.734938
min         1.000000
25%         1.000000
50%         1.000000
75%         2.000000
max         6.000000
Name: progress_speed_week1, dtype: float64


In [ ]:
# Hitung completion status akhir per enrollment (sama logika seperti sebelumnya)
progress_summary = fact_progress.groupby('enrollment_id').agg(
    modul_completed=('status', lambda x: (x == 'completed').sum())
).reset_index()

enrollment_full = fact_enrollments.merge(progress_summary, on='enrollment_id', how='left')
enrollment_full = enrollment_full.merge(dim_courses[['course_id','total_modules']], on='course_id')
enrollment_full['modul_completed'] = enrollment_full['modul_completed'].fillna(0)
enrollment_full['is_completed'] = enrollment_full['modul_completed'] >= enrollment_full['total_modules']

# Gabungkan dengan progress_speed_week1 (enrollment yang tidak punya aktivitas minggu 1 diisi 0)
enrollment_full = enrollment_full.merge(progress_speed_week1, on='enrollment_id', how='left')
enrollment_full['progress_speed_week1'] = enrollment_full['progress_speed_week1'].fillna(0)

# Bandingkan rata-rata progress_speed_week1: completer vs non-completer
print("Rata-rata progress_speed_week1 berdasarkan status akhir:")
print(enrollment_full.groupby('is_completed')['progress_speed_week1'].describe())

Rata-rata progress_speed_week1 berdasarkan status akhir:
               count      mean       std  min  25%  50%  75%  max
is_completed                                                     
False         5663.0  0.597033  0.796986  0.0  0.0  0.0  1.0  5.0
True          2284.0  1.130911  1.004122  0.0  0.0  1.0  2.0  6.0


In [ ]:
# Simpan dataset analitik lengkap (siap dipakai di Power BI juga)
output_path = '/content/drive/MyDrive/Data Analyst/EduSmart/data/processed'
import os
os.makedirs(output_path, exist_ok=True)

enrollment_full.to_csv(f'{output_path}/enrollment_analytical.csv', index=False)
print(f"Dataset analitik tersimpan: {len(enrollment_full)} baris, kolom: {list(enrollment_full.columns)}")

Dataset analitik tersimpan: 7947 baris, kolom: ['enrollment_id', 'user_id', 'course_id', 'enrollment_date', 'enrollment_source', 'date_anomaly_flag', 'is_retake', 'modul_completed', 'total_modules', 'is_completed', 'progress_speed_week1']


In [ ]:
# Dokumentasi Feature Engineering
dokumentasi_FE = """
=== RINGKASAN FEATURE ENGINEERING ===

Fitur baru: progress_speed_week1 (jumlah modul completed dalam 7 hari
pertama sejak enrollment_date).

TEMUAN PENTING (menjawab Pertanyaan Bisnis #5): kecepatan progress di
minggu pertama adalah SINYAL PREDIKTIF KUAT untuk completion akhir.
- User yang akhirnya completed: rata-rata 1.13 modul selesai di minggu 1
  (median = 1)
- User yang tidak completed: rata-rata 0.60 modul selesai di minggu 1
  (median = 0 -- lebih dari separuh tidak menyelesaikan modul apapun
  di minggu pertama)

REKOMENDASI: EduSmart dapat membangun early warning system -- jika user
belum menyelesaikan modul apapun dalam 3-5 hari sejak enrollment, picu
reminder/dorongan ekstra sebelum momentum belajar hilang sepenuhnya.

Dataset analitik (enrollment_analytical.csv) disimpan di folder
data/processed, berisi enrollment_full dengan kolom tambahan is_completed
dan progress_speed_week1 -- siap dipakai sebagai sumber data tambahan
di Power BI.
"""
print(dokumentasi_FE)

log_path = '/content/drive/MyDrive/Data Analyst/EduSmart'
with open(f'{log_path}/data_cleaning_log.txt', 'a') as f:
    f.write(dokumentasi_FE)
print("Dokumentasi Feature Engineering ditambahkan ke data_cleaning_log.txt")


=== RINGKASAN FEATURE ENGINEERING ===

Fitur baru: progress_speed_week1 (jumlah modul completed dalam 7 hari
pertama sejak enrollment_date).

TEMUAN PENTING (menjawab Pertanyaan Bisnis #5): kecepatan progress di
minggu pertama adalah SINYAL PREDIKTIF KUAT untuk completion akhir.
- User yang akhirnya completed: rata-rata 1.13 modul selesai di minggu 1
  (median = 1)
- User yang tidak completed: rata-rata 0.60 modul selesai di minggu 1
  (median = 0 -- lebih dari separuh tidak menyelesaikan modul apapun
  di minggu pertama)

REKOMENDASI: EduSmart dapat membangun early warning system -- jika user
belum menyelesaikan modul apapun dalam 3-5 hari sejak enrollment, picu
reminder/dorongan ekstra sebelum momentum belajar hilang sepenuhnya.

Dataset analitik (enrollment_analytical.csv) disimpan di folder
data/processed, berisi enrollment_full dengan kolom tambahan is_completed
dan progress_speed_week1 -- siap dipakai sebagai sumber data tambahan
di Power BI.

Dokumentasi Feature Engineering dit